# Quality Checks and Diagnostics

Quality checks in this repository are intentionally practical. They are meant to answer a few concrete questions before using the workflow outputs: do the expected tables exist, do the feature partitions have sane keys and dates, do model diagnostics support the selected model, and do the retained plots reveal obvious problems?

This notebook gathers the retained checks that are still useful for the publishable workflow. Historical model-selection files and one-off patch diagnostics are not treated as required project material here.

## Check Families

The retained quality checks fall into five groups:

- table availability checks for core generated products;
- primary feature-table summaries, including row counts, date ranges, duplicate keys, and missing fractions;
- environmental diagnostics, including distributions and predictor correlations;
- species-model diagnostics, including observed-versus-predicted summaries, residuals, feature importance, and partial dependence;
- plausibility and seascape diagnostics used to interpret the environmental-regime layer.

These checks do not prove the model is correct. They help catch broken joins, missing partitions, unexpected empty tables, scale problems, and diagnostic patterns that need interpretation.


## Core Artifact Availability

The quick validation script checks representative parquet files for selected generated table families, including the H3 grid, environmental features, static features, species-training tables, and fishing-training tables. The notebook summary below extends that idea with the main release-facing products used by the notebooks.


In [24]:
from pathlib import Path
import json
import pandas as pd
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
data_dir = Path("..") / config["paths"]["data"]
plots_dir = Path("..") / config["paths"]["plots"]

grid_resolution = config["grid"]["resolution"]
region_name = config["region"]["name"]
grid_name = f"h3_res{grid_resolution}_{region_name}.parquet"

core_artifacts = {
    "grid": data_dir / "grids" / grid_name,
    "static features": data_dir / "features" / "static" / "static.parquet",
    "environmental features, 2022": data_dir / "features" / "environmental" / "year=2022" / "part.parquet",
    "fishing effort features, 2022": data_dir / "features" / "fishing_effort" / "year=2022" / "part.parquet",
    "species presence features, 2022": data_dir / "features" / "species_presence" / "year=2022" / "part.parquet",
    "species training, 2022": data_dir / "modeling" / "species_training" / "year=2022" / "part.parquet",
    "environmental regimes, 2022": data_dir / "modeling" / "environmental_regimes" / "year=2022" / "part.parquet",
    "hybrid predictions, 2022": data_dir / "modeling" / "predictions" / "hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30" / "joint" / "year=2022" / "part.parquet",
}

pd.DataFrame(
    [
        {
            "artifact": name,
            "relative_path": path.relative_to(Path("..")),
            "status": "present" if path.exists() else "missing",
        }
        for name, path in core_artifacts.items()
    ]
)

,artifact,relative_path,status
0,grid,data/grids/h3_res6_falkland_islands.parquet,present
1,static features,data/features/static/static.parquet,present
2,"environmental features, 2022",data/features/environmental/year=2022/part.par...,present
3,"fishing effort features, 2022",data/features/fishing_effort/year=2022/part.pa...,present
4,"species presence features, 2022",data/features/species_presence/year=2022/part....,present
5,"species training, 2022",data/modeling/species_training/year=2022/part....,present
6,"environmental regimes, 2022",data/modeling/environmental_regimes/year=2022/...,present
7,"hybrid predictions, 2022",data/modeling/predictions/hybrid_presence_gate...,present


## Table Schemas and Quality Interpretation

The most useful diagnostic view is often not the raw table itself, but the table contract: which columns are present, what they mean, and whether the values look fit for purpose.

### Environmental Feature Tables

**Table:** `environmental`

**Columns:** `h3`, `date`, `sst`, `ssh`, `wind_speed`, `chl_log`, `doy_sin`, `doy_cos`, anomaly fields, and gradient fields.

**Contains:** Daily environmental values on the H3 grid, plus seasonal encodings, climatological anomalies, and neighbor gradients.

**Quality interpretation:** The retained feature QA summary checks primary environmental partitions for row counts, date coverage, duplicate `h3`/`date` keys, missing fractions, and numeric ranges. The current parquet schemas and environmental plots are used to inspect the derived variables, gradients, and correlation structure.

### Fishing-Effort Feature Tables

**Table:** `fishing_effort`

**Columns:** `h3`, `date`, `fishing_hours`, `vessel_count`.

**Contains:** Daily fishing exposure summarized on the H3 grid.

**Quality interpretation:** Feature QA checks `h3`/`date` uniqueness and missingness. Operational plots check that exposure patterns are spatially and seasonally plausible.

### Species Telemetry Support Tables

**Table:** `species_presence`

**Columns:** `h3`, `date`, `species`, `presence_count`, `individual_count`, `trip_count`.

**Contains:** Telemetry support aggregated by `h3`, day, and `species`.

**Quality interpretation:** The keys include species, so zero-support cells are created later for modeling. Counts should be interpreted as telemetry support, not true abundance.

### Model Training Tables

**Table:** `species_training`

**Columns:** `h3`, `date`, `species`, `residence_index`, telemetry counts, environmental features, static features, and spatial encodings.

**Contains:** Model-ready species-use rows combining telemetry support with environmental and static predictors.

**Quality interpretation:** This is the main training table. Duplicate `h3`/`date`/`species` keys would be a serious issue. Core availability checks, schema inspection, validation metrics, and model diagnostics are the retained checks for this table family.

### Environmental-Regime Tables

**Table:** `environmental_regimes`

**Columns:** `h3`, `date`, `som_prototype`, `seascape`, `seascape_distance`, and Bayesian GMM component fields.

**Contains:** Daily H3 environmental-regime labels from the selected SOM-hierarchical seascapes and Bayesian GMM components.

**Quality interpretation:** The compact summary checks coverage, number of classes/components, assignment distances, component probabilities, and missing assignment rows.

### Prediction Tables

**Table:** `hybrid_predictions`

**Columns:** `h3`, `date`, `species`, `species_use_log_pred`, `hybrid_alpha`, `species_use_ml_log_pred`, `plausibility`, `plausibility_gate`, `fishing_activity_log`, `risk_log_pred`.

**Contains:** Final daily species-use and realized-risk prediction surfaces with the intermediate plausibility-gate fields retained for interpretation.

**Quality interpretation:** Prediction diagnostics focus on relative interpretation. Plausibility is environmental support, not presence probability.

### Weekly Operator Tables

**Table:** `weekly_operator`

**Columns:** `h3`, `species`, `iso_week`, `n_days`, `species_use_log_pred_mean`, `latent_risk_log_pred_mean`, and `display_latent_risk_log_pred_mean`. The by-year and sequence products also include `iso_year`; the climatology product includes `n_years`.

**Contains:** Weekly latent-risk summaries for operational maps and animations.

**Quality interpretation:** The display field intentionally suppresses very low species-use cells. Weekly maps check whether the summarized product remains interpretable.


## Feature QA Summary

The feature QA summary is written to `data/processed/feature_qa_summary.json`. It summarizes the primary feature tables: environmental features, fishing-effort features, species-presence support, and static features. It records whether tables exist, available partitions, row counts, unique H3 cells, unique dates, duplicate key rows, missing fractions, and numeric ranges.

The most important field to scan first is `duplicate_key_rows`. For yearly feature tables, duplicate `h3`/`date` keys, or duplicate `h3`/`date`/`species` keys in species-presence support, would indicate a join or aggregation problem.


In [25]:
summary_path = data_dir / "processed" / "feature_qa_summary.json"
feature_summary = json.loads(summary_path.read_text())

rows = []
for table_name, table_summary in feature_summary.items():
    year_rows = table_summary.get("years", [])
    if not year_rows:
        rows.append(
            {
                "table": table_name,
                "years": "none",
                "n_partitions": 0,
                "total_rows": 0,
                "min_unique_h3": None,
                "max_unique_h3": None,
                "duplicate_key_rows": None,
                "max_missing_fraction": None,
            }
        )
        continue

    max_missing = []
    for year_summary in year_rows:
        missing = year_summary.get("missing_fraction", {})
        if missing:
            max_missing.append(max(missing.values()))

    rows.append(
        {
            "table": table_name,
            "years": f"{year_rows[0].get('year')}-{year_rows[-1].get('year')}",
            "n_partitions": len(year_rows),
            "total_rows": sum(item.get("rows", 0) or 0 for item in year_rows),
            "min_unique_h3": min(
                (item.get("unique_h3") for item in year_rows if item.get("unique_h3") is not None),
                default=None,
            ),
            "max_unique_h3": max(
                (item.get("unique_h3") for item in year_rows if item.get("unique_h3") is not None),
                default=None,
            ),
            "duplicate_key_rows": sum(item.get("duplicate_key_rows", 0) or 0 for item in year_rows),
            "max_missing_fraction": round(max(max_missing), 4) if max_missing else None,
        }
    )

pd.DataFrame(rows).sort_values("table")


,table,years,n_partitions,total_rows,min_unique_h3,max_unique_h3,duplicate_key_rows,max_missing_fraction
0,environmental,2014-2023,10,135887268,37209.0,37209.0,0.0,0.034
1,fishing_effort,2014-2023,10,849818,8447.0,9876.0,0.0,0.000
2,species_presence,2022-2023,2,10268,1874.0,5335.0,0.0,0.000
3,static,none,0,0,NaN,NaN,NaN,NaN


In [26]:
issue_rows = []
for table_name, table_summary in feature_summary.items():
    for year_summary in table_summary.get("years", []):
        duplicate_rows = year_summary.get("duplicate_key_rows", 0) or 0
        missing = year_summary.get("missing_fraction", {})
        high_missing = {
            key: value
            for key, value in missing.items()
            if value is not None and value > 0.01
        }
        if duplicate_rows or high_missing:
            issue_rows.append(
                {
                    "table": table_name,
                    "year": year_summary.get("year"),
                    "duplicate_key_rows": duplicate_rows,
                    "columns_with_missing_gt_1pct": ", ".join(sorted(high_missing)) or "none",
                    "max_missing_fraction": round(max(high_missing.values()), 4) if high_missing else 0.0,
                }
            )

pd.DataFrame(issue_rows).sort_values(["table", "year"])


,table,year,duplicate_key_rows,columns_with_missing_gt_1pct,max_missing_fraction
0,environmental,2014,0,"chl, wind_u10, wind_v10",0.0340
1,environmental,2015,0,"chl, wind_u10, wind_v10",0.0340
2,environmental,2016,0,"chl, wind_u10, wind_v10",0.0339
3,environmental,2017,0,"chl, wind_u10, wind_v10",0.0340
4,environmental,2018,0,"chl, wind_u10, wind_v10",0.0340
5,environmental,2019,0,"chl, wind_u10, wind_v10",0.0340
6,environmental,2020,0,"chl, wind_u10, wind_v10",0.0339
7,environmental,2021,0,"chl, wind_u10, wind_v10",0.0340
8,environmental,2022,0,"chl, wind_u10, wind_v10",0.0340
9,environmental,2023,0,"chl, wind_u10, wind_v10",0.0340


## Environmental Diagnostics

Environmental diagnostics are used to inspect variable distributions, anomaly behavior, and predictor correlations before those variables enter the model.

<p>
  <img src="../plots/environmental_histograms/sst_anom_histogram_2014-2023.png" width="49%" />
  <img src="../plots/environmental_histograms/chl_log_anom_histogram_2014-2023.png" width="49%" />
</p>

<p>
  <img src="../plots/environmental_correlation/environmental_predictors_spearman_correlation_2022.png" width="70%" />
</p>


## Historical Model Comparison

A small set of historical model-comparison diagnostics is still retained for context. These metrics compare observed versus predicted species-use values for earlier candidate learners using the retained diagnostic exports. They are useful for understanding why Extra Trees became the active species-use learner, but they should not be confused with the selected SOM k=30 grouped cross-validation summary shown later.


In [27]:
comparison_files = {
    "Extra Trees": data_dir / "plot_exports" / "species_use_diagnostics" / "extra_trees_observed_vs_predicted_metrics.csv",
    "Random Forest": data_dir / "plot_exports" / "species_use_diagnostics" / "random_forest_observed_vs_predicted_metrics.csv",
    "Histogram Gradient Boosting": data_dir / "plot_exports" / "species_use_diagnostics" / "hist_gradient_boosting_observed_vs_predicted_metrics.csv",
}

comparison_rows = []
for model_name, path in comparison_files.items():
    row = pd.read_csv(path).iloc[0]
    comparison_rows.append(
        {
            "model": model_name,
            "R2": round(row["r2"], 3),
            "RMSE": round(row["rmse"], 3),
            "MAE": round(row["mae"], 3),
        }
    )

pd.DataFrame(comparison_rows).sort_values("R2", ascending=False)


,model,R2,RMSE,MAE
0,Extra Trees,0.916,27.758,2.379
1,Random Forest,0.807,42.186,3.274
2,Histogram Gradient Boosting,0.606,60.273,3.665


## Model Diagnostics

The retained model diagnostics focus on the active Extra Trees species-use learner. The observed-versus-predicted and residual plots help evaluate whether the model captures the broad signal without treating the scores as absolute probabilities. Feature importance and partial dependence are interpretation aids, not causal claims.

<p>
  <img src="../plots/species_use_diagnostics/extra_trees_observed_vs_predicted.png" width="49%" />
  <img src="../plots/species_use_diagnostics/extra_trees_log_residual_distribution.png" width="49%" />
</p>

<p>
  <img src="../plots/species_use_diagnostics/species_feature_importance_top_features.png" width="49%" />
  <img src="../plots/species_use_diagnostics/extra_trees_partial_dependence.png" width="49%" />
</p>


In [28]:
metrics_path = data_dir / "plot_exports" / "species_use_diagnostics" / "extra_trees_observed_vs_predicted_metrics.csv"
metrics = pd.read_csv(metrics_path)
metrics

,r2,rmse,mae
0,0.916454,27.757684,2.37949


## Validation and Plausibility Diagnostics

The selected validation summary is retained under `data/modeling/metrics/`. It reports the SOM-hierarchical k=30 grouped cross-validation summary used to support the active Extra Trees species-use learner. Plausibility diagnostics are retained separately because the Bayesian GMM layer is used as environmental support, not as the primary species-use model.

<p>
  <img src="../plots/plausibility/yearly_non_zero_median_plausibility_2014-2023.png" width="49%" />
  <img src="../plots/predictions/hybrid_presence_gate_extra_trees_som_hierarchical_k30_5fold_blockcv_bayesian_gmm_k30_joint_latent_risk_plausibility_gate_sensitivity_2022_small_multiples.png" width="49%" />
</p>


In [29]:
yearly_plausibility = pd.read_csv(
    data_dir / "plot_exports" / "plausibility" / "yearly_plausibility_2014-2023.csv"
)

(
    yearly_plausibility.groupby("species", as_index=False)
    .agg(
        years=("year", "nunique"),
        median_non_zero_plausibility=("non_zero_median_plausibility", "median"),
        min_non_zero_plausibility=("non_zero_median_plausibility", "min"),
        max_non_zero_plausibility=("non_zero_median_plausibility", "max"),
        total_non_zero_cell_days=("non_zero_cell_days", "sum"),
    )
)


,species,years,median_non_zero_plausibility,min_non_zero_plausibility,max_non_zero_plausibility,total_non_zero_cell_days
0,BBAL,10,0.245446,0.203444,0.559142,443741.0
1,SAFS,10,0.169218,0.133050,0.242617,2774630.0


In [30]:
validation_metrics = pd.read_csv(
    data_dir / "modeling" / "metrics" / "species_model_block_cv_selected_som_k30_5fold.csv"
)

row = validation_metrics.iloc[0]
pd.DataFrame(
    {
        "metric": [
            "validation variant",
            "model",
            "CV folds",
            "R2 mean",
            "R2 std",
            "log R2 mean",
            "log RMSE mean",
            "mean test fraction",
        ],
        "value": [
            row["validation_variant"],
            row["model"],
            int(row["cv_folds"]),
            round(row["r2_mean"], 3),
            round(row["r2_std"], 3),
            round(row["r2_log_mean"], 3),
            round(row["rmse_log_mean"], 3),
            round(row["actual_test_fraction_mean"], 3),
        ],
    }
)


,metric,value
0,validation variant,som_k30_5fold
1,model,extra_trees
2,CV folds,5
3,R2 mean,0.769
4,R2 std,0.22
5,log R2 mean,0.518
6,log RMSE mean,0.538
7,mean test fraction,0.2


## Seascape and Environmental-Regime Diagnostics

The selected environmental-regime table combines SOM-hierarchical k=30 seascape assignments with Bayesian GMM component assignments. The first summary checks table coverage and assignment fields without printing the full `h3`/`date` table. The second summary checks whether the selected seascape classes are represented and whether any class has very low support.


In [31]:
import duckdb

regime_glob = data_dir / "modeling" / "environmental_regimes" / "year=*" / "part.parquet"
query = f"""
    SELECT
        COUNT(*)::BIGINT AS rows,
        COUNT(DISTINCT h3)::BIGINT AS unique_h3,
        MIN(date)::DATE AS date_min,
        MAX(date)::DATE AS date_max,
        COUNT(DISTINCT seascape)::INTEGER AS seascape_classes,
        COUNT(DISTINCT som_prototype)::INTEGER AS som_prototypes,
        COUNT(DISTINCT bayesian_gmm_k30_component)::INTEGER AS bayesian_gmm_components,
        AVG(seascape_distance)::DOUBLE AS mean_seascape_distance,
        MAX(seascape_distance)::DOUBLE AS max_seascape_distance,
        AVG(bayesian_gmm_k30_component_probability)::DOUBLE AS mean_component_probability,
        AVG(bayesian_gmm_k30_component_entropy)::DOUBLE AS mean_component_entropy,
        SUM(CASE WHEN seascape IS NULL THEN 1 ELSE 0 END)::BIGINT AS missing_seascape_rows,
        SUM(CASE WHEN bayesian_gmm_k30_component IS NULL THEN 1 ELSE 0 END)::BIGINT AS missing_component_rows
    FROM read_parquet('{regime_glob}', hive_partitioning=false)
"""

with duckdb.connect(database=":memory:") as con:
    environmental_regime_summary = con.execute(query).fetchdf()

environmental_regime_summary.T.rename(columns={0: "value"})


,value
rows,128958431
unique_h3,36349
date_min,2014-01-01 00:00:00
date_max,2023-12-31 00:00:00
seascape_classes,30
som_prototypes,225
bayesian_gmm_components,30
mean_seascape_distance,2.434754
max_seascape_distance,50.177681
mean_component_probability,0.997848


In [32]:
seascape_summary = pd.read_csv(
    data_dir / "modeling" / "metrics" / "seascapes" / "seascape_summary_som_15x15_hierarchical_k30_2014-2023.csv"
)

seascape_summary_summary = pd.DataFrame(
    {
        "metric": [
            "number of seascape classes",
            "total H3/date assignments",
            "minimum class cell-days",
            "maximum class cell-days",
            "minimum SOM prototypes per class",
            "maximum SOM prototypes per class",
        ],
        "value": [
            seascape_summary["seascape"].nunique(),
            int(seascape_summary["n_cell_days"].sum()),
            int(seascape_summary["n_cell_days"].min()),
            int(seascape_summary["n_cell_days"].max()),
            int(seascape_summary["n_som_prototypes"].min()),
            int(seascape_summary["n_som_prototypes"].max()),
        ],
    }
)
seascape_summary_summary


,metric,value
0,number of seascape classes,30
1,total H3/date assignments,128958431
2,minimum class cell-days,1715312
3,maximum class cell-days,8423419
4,minimum SOM prototypes per class,3
5,maximum SOM prototypes per class,15


## Regenerating Checks

The checks and diagnostic plots can be regenerated from the retained scripts. The commands below are a rebuild checklist; displaying this cell does not run them.

The first two commands regenerate tabular QA summaries. The plot groups regenerate the environmental, species-use, relationship, prediction, and seascape diagnostic figures shown or referenced above.


In [33]:
commands = [
    "python3 scripts/qa/quick_validation.py",
    "python3 scripts/qa/feature_qa_summary.py",
    "python3 scripts/plots/plot_all_maps.py --group environmental",
    "python3 scripts/plots/plot_all_maps.py --group species",
    "python3 scripts/plots/plot_all_maps.py --group diagnostics",
    "python3 scripts/plots/plot_all_maps.py --group predictions",
    "python3 scripts/plots/plot_all_maps.py --group seascapes",
]

pd.DataFrame({"command": commands})

,command
0,python3 scripts/qa/quick_validation.py
1,python3 scripts/qa/feature_qa_summary.py
2,python3 scripts/plots/plot_all_maps.py --group...
3,python3 scripts/plots/plot_all_maps.py --group...
4,python3 scripts/plots/plot_all_maps.py --group...
5,python3 scripts/plots/plot_all_maps.py --group...
6,python3 scripts/plots/plot_all_maps.py --group...
